## 🤖 **LLM-Based Outlier Filtering for VAE Data**

Before feeding data to the VAE, we'll implement an intelligent outlier detection system using Large Language Models (LLMs). This approach leverages semantic understanding to identify anomalous samples that could negatively impact VAE training.

### 🎯 **Filtering Strategies**
1. **Image Description Analysis**: Convert images to text descriptions and analyze semantic consistency
2. **Statistical Anomaly Detection**: Combine traditional metrics with LLM insights
3. **Multi-modal Filtering**: Use both visual and textual representations
4. **Confidence-based Selection**: Filter based on LLM confidence scores
5. **Semantic Clustering**: Group similar samples and identify outliers

In [ ]:
# 🎛️ CONFIGURE AND APPLY LLM OUTLIER FILTERING

# Configuration for outlier filtering
FILTER_CONFIG = {
    'model_type': 'local',  # Use local analysis (no API keys needed)
    'outlier_threshold': 0.2,  # Higher = more strict filtering
    'confidence_threshold': 0.6,  # Minimum confidence to keep sample
    'batch_size': 32,  # Process in batches for efficiency
    'enable_filtering': True,  # Set to False to disable filtering
}

def apply_llm_filtering_to_dataset(train_loader, test_loader, config=FILTER_CONFIG):
    """
    Apply LLM-based outlier filtering to training and test datasets.
    
    Args:
        train_loader: Training DataLoader
        test_loader: Test DataLoader  
        config: Filtering configuration
        
    Returns:
        Tuple of (filtered_train_data, filtered_test_data, filtering_stats)
    """
    
    if not config['enable_filtering']:
        print("🔄 LLM filtering disabled - using original datasets")
        return train_loader, test_loader, {'filtering_enabled': False}
    
    print("🤖 Applying LLM-based outlier filtering to datasets...")
    
    # Initialize the filter
    filter_system = LLMOutlierFilter(
        model_type=config['model_type'],
        outlier_threshold=config['outlier_threshold'],
        confidence_threshold=config['confidence_threshold'],
        batch_size=config['batch_size']
    )
    
    # Process training data
    print("\n📚 Filtering training dataset...")
    filtered_train_images = []
    filtered_train_labels = []
    train_stats = []
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        if batch_idx >= 5:  # Limit to first 5 batches for demo
            break
            
        print(f"Processing training batch {batch_idx + 1}...")
        
        # Apply filtering
        filt_imgs, filt_labels, stats = filter_system.filter_batch(images, labels)
        
        filtered_train_images.append(filt_imgs)
        filtered_train_labels.append(filt_labels)
        train_stats.append(stats)
    
    # Process test data (sample)
    print("\n🧪 Filtering test dataset (sample)...")
    filtered_test_images = []
    filtered_test_labels = []
    test_stats = []
    
    for batch_idx, (images, labels) in enumerate(test_loader):
        if batch_idx >= 2:  # Limit to first 2 batches for demo
            break
            
        print(f"Processing test batch {batch_idx + 1}...")
        
        # Apply filtering
        filt_imgs, filt_labels, stats = filter_system.filter_batch(images, labels)
        
        filtered_test_images.append(filt_imgs)
        filtered_test_labels.append(filt_labels)
        test_stats.append(stats)
    
    # Combine filtered data
    if filtered_train_images:
        train_images_combined = torch.cat(filtered_train_images, dim=0)
        train_labels_combined = torch.cat(filtered_train_labels, dim=0)
    else:
        train_images_combined = torch.empty(0, 1, 28, 28)
        train_labels_combined = torch.empty(0, dtype=torch.long)
    
    if filtered_test_images:
        test_images_combined = torch.cat(filtered_test_images, dim=0)
        test_labels_combined = torch.cat(filtered_test_labels, dim=0)
    else:
        test_images_combined = torch.empty(0, 1, 28, 28)
        test_labels_combined = torch.empty(0, dtype=torch.long)
    
    # Calculate overall statistics
    train_original = sum(stats['original_count'] for stats in train_stats)
    train_filtered = sum(stats['filtered_count'] for stats in train_stats)
    test_original = sum(stats['original_count'] for stats in test_stats)
    test_filtered = sum(stats['filtered_count'] for stats in test_stats)
    
    overall_stats = {
        'filtering_enabled': True,
        'train_original_count': train_original,
        'train_filtered_count': train_filtered,
        'train_removal_rate': (train_original - train_filtered) / train_original if train_original > 0 else 0,
        'test_original_count': test_original,
        'test_filtered_count': test_filtered,
        'test_removal_rate': (test_original - test_filtered) / test_original if test_original > 0 else 0,
        'average_outlier_score': np.mean([stats['average_outlier_score'] for stats in train_stats + test_stats]),
        'average_confidence': np.mean([stats['average_confidence'] for stats in train_stats + test_stats]),
        'config': config
    }
    
    print(f"\n📊 Filtering Summary:")
    print(f"Training: {train_filtered}/{train_original} samples kept ({overall_stats['train_removal_rate']*100:.1f}% removed)")
    print(f"Test: {test_filtered}/{test_original} samples kept ({overall_stats['test_removal_rate']*100:.1f}% removed)")
    print(f"Average outlier score: {overall_stats['average_outlier_score']:.3f}")
    print(f"Average confidence: {overall_stats['average_confidence']:.3f}")
    
    # Create new datasets (simplified - in practice you'd create proper DataLoaders)
    filtered_data = {
        'train_images': train_images_combined,
        'train_labels': train_labels_combined,
        'test_images': test_images_combined,
        'test_labels': test_labels_combined
    }
    
    return filtered_data, overall_stats

def visualize_filtering_results(original_images, filtered_images, analysis_results, max_display=8):
    """
    Visualize the results of LLM filtering to show which samples were removed.
    
    Args:
        original_images: Original batch of images
        filtered_images: Filtered batch of images  
        analysis_results: Results from the filtering analysis
        max_display: Maximum number of images to display
    """
    
    print("🎨 Visualizing LLM filtering results...")
    
    keep_indices = analysis_results['keep_indices']
    outlier_scores = analysis_results['outlier_scores']
    confidence_scores = analysis_results['confidence_scores']
    descriptions = analysis_results['descriptions']
    
    # Show kept vs removed samples
    n_original = len(original_images)
    n_show = min(max_display, n_original)
    
    fig, axes = plt.subplots(3, n_show, figsize=(2*n_show, 6))
    if n_show == 1:
        axes = axes.reshape(-1, 1)
    
    for i in range(n_show):
        # Original image
        axes[0, i].imshow(original_images[i].squeeze(), cmap='gray')
        kept_status = "✅ KEPT" if i in keep_indices else "❌ REMOVED"
        axes[0, i].set_title(f'{kept_status}\nOutlier: {outlier_scores[i]:.2f}', fontsize=8)
        axes[0, i].axis('off')
        
        # Confidence and description info
        axes[1, i].text(0.1, 0.8, f'Confidence: {confidence_scores[i]:.2f}', 
                       transform=axes[1, i].transAxes, fontsize=8)
        axes[1, i].text(0.1, 0.6, f'Description:', transform=axes[1, i].transAxes, fontsize=8, weight='bold')
        axes[1, i].text(0.1, 0.4, descriptions[i][:20] + ('...' if len(descriptions[i]) > 20 else ''), 
                       transform=axes[1, i].transAxes, fontsize=7, wrap=True)
        axes[1, i].axis('off')
        
        # Score visualization
        bar_height = 0.3
        axes[2, i].barh(0, outlier_scores[i], height=bar_height, color='red', alpha=0.7, label='Outlier Score')
        axes[2, i].barh(0.4, confidence_scores[i], height=bar_height, color='green', alpha=0.7, label='Confidence')
        axes[2, i].set_xlim(0, 1)
        axes[2, i].set_ylim(-0.2, 0.8)
        axes[2, i].set_xlabel('Score', fontsize=8)
        if i == 0:
            axes[2, i].legend(fontsize=6)
        axes[2, i].tick_params(labelsize=6)
    
    plt.suptitle('🤖 LLM-Based Outlier Filtering Results', fontsize=14, weight='bold')
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print(f"\n📈 Filtering Statistics:")
    print(f"Original samples: {n_original}")
    print(f"Kept samples: {len(keep_indices)}")
    print(f"Removed samples: {n_original - len(keep_indices)}")
    print(f"Removal rate: {((n_original - len(keep_indices)) / n_original * 100):.1f}%")
    print(f"Average outlier score: {np.mean(outlier_scores):.3f}")
    print(f"Average confidence: {np.mean(confidence_scores):.3f}")

def analyze_filtering_impact_on_vae(original_data, filtered_data, model, device):
    """
    Analyze how LLM filtering affects VAE training quality.
    
    Args:
        original_data: Original training data
        filtered_data: LLM-filtered training data
        model: Trained VAE model
        device: Computing device
        
    Returns:
        Dictionary with impact analysis results
    """
    
    print("🔬 Analyzing LLM filtering impact on VAE performance...")
    
    model.eval()
    
    # Sample from both datasets for comparison
    n_samples = min(100, len(original_data), len(filtered_data['train_images']))
    
    if n_samples == 0:
        print("⚠️ Insufficient data for impact analysis")
        return {'status': 'insufficient_data'}
    
    # Get samples
    orig_samples = original_data[:n_samples].to(device)
    filt_samples = filtered_data['train_images'][:n_samples].to(device)
    
    with torch.no_grad():
        # Analyze original data
        orig_outputs = model(orig_samples)
        orig_recon = orig_outputs[7]  # Final reconstruction
        orig_latent_mu = orig_outputs[5]  # Latent means
        
        # Analyze filtered data  
        filt_outputs = model(filt_samples)
        filt_recon = filt_outputs[7]
        filt_latent_mu = filt_outputs[5]
        
        # Calculate metrics
        orig_recon_loss = F.mse_loss(orig_samples, orig_recon)
        filt_recon_loss = F.mse_loss(filt_samples, filt_recon)
        
        # Latent space analysis
        orig_latent_std = orig_latent_mu.std(dim=0).mean()
        filt_latent_std = filt_latent_mu.std(dim=0).mean()
        
        # Content vs transform latent analysis
        orig_content_std = orig_latent_mu[:, :2].std(dim=0).mean()
        orig_transform_std = orig_latent_mu[:, 2:].std(dim=0).mean()
        filt_content_std = filt_latent_mu[:, :2].std(dim=0).mean()
        filt_transform_std = filt_latent_mu[:, 2:].std(dim=0).mean()
    
    # Impact analysis
    impact_results = {
        'status': 'completed',
        'reconstruction_quality': {
            'original_mse': orig_recon_loss.item(),
            'filtered_mse': filt_recon_loss.item(),
            'improvement': (orig_recon_loss - filt_recon_loss).item(),
            'improvement_percent': ((orig_recon_loss - filt_recon_loss) / orig_recon_loss * 100).item()
        },
        'latent_space_diversity': {
            'original_std': orig_latent_std.item(),
            'filtered_std': filt_latent_std.item(),
            'diversity_change': (filt_latent_std - orig_latent_std).item()
        },
        'structured_latent_analysis': {
            'content_diversity_original': orig_content_std.item(),
            'content_diversity_filtered': filt_content_std.item(),
            'transform_diversity_original': orig_transform_std.item(), 
            'transform_diversity_filtered': filt_transform_std.item()
        }
    }
    
    print(f"📊 Impact Analysis Results:")
    print(f"Reconstruction Quality:")
    print(f"  Original MSE: {impact_results['reconstruction_quality']['original_mse']:.6f}")
    print(f"  Filtered MSE: {impact_results['reconstruction_quality']['filtered_mse']:.6f}")
    print(f"  Improvement: {impact_results['reconstruction_quality']['improvement_percent']:.2f}%")
    
    print(f"Latent Space Diversity:")
    print(f"  Original std: {impact_results['latent_space_diversity']['original_std']:.4f}")
    print(f"  Filtered std: {impact_results['latent_space_diversity']['filtered_std']:.4f}")
    
    if impact_results['reconstruction_quality']['improvement'] > 0:
        print("✅ LLM filtering improved reconstruction quality!")
    elif impact_results['reconstruction_quality']['improvement'] > -0.001:
        print("➡️ LLM filtering had minimal impact on reconstruction quality")
    else:
        print("⚠️ LLM filtering may have reduced reconstruction quality")
    
    return impact_results

print("✅ LLM outlier filtering system ready!")
print("🎯 Ready to filter datasets and analyze impact on VAE training")

In [ ]:
# 🚀 DEMONSTRATE LLM FILTERING ON MNIST DATA

print("🎯 Demonstrating LLM-based outlier filtering on MNIST data...")

# First, let's create some artificial outliers to test the system
def inject_outliers(images, labels, outlier_ratio=0.1):
    """
    Inject artificial outliers into the dataset to test filtering effectiveness.
    
    Args:
        images: Original images tensor
        labels: Original labels tensor
        outlier_ratio: Fraction of data to replace with outliers
        
    Returns:
        Tuple of (corrupted_images, corrupted_labels, outlier_indices)
    """
    n_samples = len(images)
    n_outliers = int(n_samples * outlier_ratio)
    
    # Choose random indices for outliers
    outlier_indices = torch.randperm(n_samples)[:n_outliers]
    
    corrupted_images = images.clone()
    corrupted_labels = labels.clone()
    
    print(f"💥 Injecting {n_outliers} artificial outliers...")
    
    for i, idx in enumerate(outlier_indices):
        outlier_type = i % 4  # 4 different types of outliers
        
        if outlier_type == 0:
            # Random noise image
            corrupted_images[idx] = torch.rand_like(images[idx])
            
        elif outlier_type == 1:
            # All white image
            corrupted_images[idx] = torch.ones_like(images[idx])
            
        elif outlier_type == 2:
            # All black image
            corrupted_images[idx] = torch.zeros_like(images[idx])
            
        elif outlier_type == 3:
            # Checkerboard pattern
            img = torch.zeros_like(images[idx])
            for y in range(0, 28, 4):
                for x in range(0, 28, 4):
                    if (y // 4 + x // 4) % 2 == 0:
                        img[0, y:min(y+4, 28), x:min(x+4, 28)] = 1.0
            corrupted_images[idx] = img
    
    print(f"✅ Outliers injected at indices: {outlier_indices.tolist()}")
    
    return corrupted_images, corrupted_labels, outlier_indices.tolist()

# Test the filtering system
if 'test_samples' in globals() and len(test_samples) > 0:
    # Use existing test samples
    print("📊 Using existing test samples for demonstration...")
    demo_images = test_samples[:32].clone()  # Take first 32 samples
    demo_labels = test_labels[:32].clone() if 'test_labels' in globals() else torch.randint(0, 10, (32,))
    
    # Inject some artificial outliers
    corrupted_images, corrupted_labels, outlier_indices = inject_outliers(demo_images, demo_labels, outlier_ratio=0.25)
    
    print(f"\n🧪 Testing LLM filtering on {len(corrupted_images)} samples (with {len(outlier_indices)} artificial outliers)...")
    
    # Apply LLM filtering
    filter_system = LLMOutlierFilter(
        model_type='local',  # Use local analysis
        outlier_threshold=0.15,  # Moderate threshold
        confidence_threshold=0.6,  # Moderate confidence requirement
        batch_size=32
    )
    
    # Filter the corrupted data
    filtered_images, filtered_labels, analysis_results = filter_system.filter_batch(
        corrupted_images, corrupted_labels
    )
    
    print(f"\n📈 Filtering Performance:")
    print(f"Original samples: {len(corrupted_images)}")
    print(f"Artificial outliers injected: {len(outlier_indices)}")
    print(f"Samples after filtering: {len(filtered_images)}")
    print(f"Outliers detected: {len(corrupted_images) - len(filtered_images)}")
    
    # Check how many actual outliers were caught
    detected_outliers = set(range(len(corrupted_images))) - set(analysis_results['keep_indices'])
    actual_outliers_caught = len(detected_outliers.intersection(set(outlier_indices)))
    
    print(f"Actual outliers successfully detected: {actual_outliers_caught}/{len(outlier_indices)} ({100*actual_outliers_caught/len(outlier_indices):.1f}%)")
    
    # Visualize results
    visualize_filtering_results(corrupted_images, filtered_images, analysis_results, max_display=8)
    
    # Store results for further analysis
    llm_filtering_demo_results = {
        'original_images': demo_images,
        'corrupted_images': corrupted_images,
        'filtered_images': filtered_images,
        'outlier_indices': outlier_indices,
        'analysis_results': analysis_results,
        'detection_accuracy': actual_outliers_caught / len(outlier_indices) if outlier_indices else 0
    }
    
    print(f"\n✅ LLM filtering demonstration completed!")
    print(f"🎯 Detection accuracy: {llm_filtering_demo_results['detection_accuracy']*100:.1f}%")
    
else:
    print("⚠️ No test samples available for demonstration")
    print("Please run the data loading cells first to load MNIST data")
    
    # Create synthetic MNIST-like data for demo
    print("🔄 Creating synthetic data for demonstration...")
    
    # Generate simple synthetic digit-like images
    demo_images = []
    for i in range(16):
        img = torch.zeros(1, 28, 28)
        
        # Create simple digit-like patterns
        digit = i % 10
        
        if digit == 0:  # Circle
            center = 14
            for y in range(28):
                for x in range(28):
                    if 8 <= np.sqrt((x-center)**2 + (y-center)**2) <= 12:
                        img[0, y, x] = 1.0
        elif digit == 1:  # Vertical line
            img[0, 5:23, 12:16] = 1.0
        # Add more patterns as needed
        else:
            # Simple rectangular pattern for other digits
            img[0, 8:20, 6:22] = 1.0
            
        demo_images.append(img)
    
    demo_images = torch.stack(demo_images)
    demo_labels = torch.tensor([i % 10 for i in range(16)])
    
    print(f"📊 Created {len(demo_images)} synthetic samples")
    
    # Apply filtering to synthetic data
    filter_system = LLMOutlierFilter(model_type='local')
    filtered_images, filtered_labels, analysis_results = filter_system.filter_batch(demo_images, demo_labels)
    
    print(f"Filtered {len(demo_images)} → {len(filtered_images)} samples")
    
    # Store synthetic results
    llm_filtering_demo_results = {
        'synthetic_data': True,
        'original_images': demo_images,
        'filtered_images': filtered_images,
        'analysis_results': analysis_results
    }

In [ ]:
# 🔗 INTEGRATE LLM FILTERING WITH VAE TRAINING PIPELINE

def create_filtered_dataloader(original_loader, filter_config, sample_limit=None):
    """
    Create a new DataLoader with LLM-filtered data.
    
    Args:
        original_loader: Original DataLoader
        filter_config: LLM filtering configuration
        sample_limit: Limit number of batches to process (for demo)
        
    Returns:
        New DataLoader with filtered data
    """
    from torch.utils.data import TensorDataset, DataLoader
    
    print(f"🔄 Creating filtered DataLoader...")
    
    # Initialize filter
    filter_system = LLMOutlierFilter(**filter_config)
    
    # Collect filtered data
    all_filtered_images = []
    all_filtered_labels = []
    total_original = 0
    total_filtered = 0
    
    for batch_idx, (images, labels) in enumerate(original_loader):
        if sample_limit and batch_idx >= sample_limit:
            break
            
        # Apply filtering
        filtered_imgs, filtered_lbls, stats = filter_system.filter_batch(images, labels)
        
        if len(filtered_imgs) > 0:
            all_filtered_images.append(filtered_imgs)
            all_filtered_labels.append(filtered_lbls)
        
        total_original += stats['original_count']
        total_filtered += stats['filtered_count']
        
        if (batch_idx + 1) % 5 == 0:
            print(f"   Processed {batch_idx + 1} batches...")
    
    if not all_filtered_images:
        print("⚠️ No data passed filtering! Using original data.")
        return original_loader
    
    # Combine all filtered data
    combined_images = torch.cat(all_filtered_images, dim=0)
    combined_labels = torch.cat(all_filtered_labels, dim=0)
    
    # Create new dataset and dataloader
    filtered_dataset = TensorDataset(combined_images, combined_labels)
    filtered_loader = DataLoader(
        filtered_dataset, 
        batch_size=original_loader.batch_size,
        shuffle=True
    )
    
    print(f"✅ Filtered DataLoader created:")
    print(f"   Original samples: {total_original}")
    print(f"   Filtered samples: {total_filtered}")
    print(f"   Reduction: {100*(total_original-total_filtered)/total_original:.1f}%")
    
    return filtered_loader

def train_vae_with_llm_filtering(model, train_loader, test_loader, config):
    """
    Train VAE with LLM-filtered data and compare performance.
    
    Args:
        model: VAE model to train
        train_loader: Training DataLoader
        test_loader: Test DataLoader
        config: Training configuration
        
    Returns:
        Training results with filtering comparison
    """
    
    print("🎯 Training VAE with LLM-filtered data...")
    
    # Configure LLM filtering
    filter_config = {
        'model_type': 'local',
        'outlier_threshold': 0.15,
        'confidence_threshold': 0.6,
        'batch_size': 32
    }
    
    # Create filtered training data (limited for demo)
    print("\n📚 Creating filtered training dataset...")
    filtered_train_loader = create_filtered_dataloader(
        train_loader, 
        filter_config, 
        sample_limit=3  # Limit for demonstration
    )
    
    # Optional: Create filtered test data
    print("\n🧪 Creating filtered test dataset...")
    filtered_test_loader = create_filtered_dataloader(
        test_loader, 
        filter_config, 
        sample_limit=2  # Limit for demonstration
    )
    
    # Compare model performance on original vs filtered data
    print("\n🔬 Comparing model performance...")
    
    model.eval()
    
    # Sample from both datasets
    with torch.no_grad():
        # Original data performance
        orig_batch = next(iter(train_loader))
        orig_images, orig_labels = orig_batch[0][:16], orig_batch[1][:16]
        orig_images = orig_images.to(device)
        
        orig_outputs = model(orig_images)
        orig_recon_loss = F.mse_loss(orig_images, orig_outputs[7])  # Final reconstruction
        
        # Filtered data performance
        if len(filtered_train_loader) > 0:
            filt_batch = next(iter(filtered_train_loader))
            filt_images, filt_labels = filt_batch[0][:16], filt_batch[1][:16]
            filt_images = filt_images.to(device)
            
            filt_outputs = model(filt_images)
            filt_recon_loss = F.mse_loss(filt_images, filt_outputs[7])
            
            # Performance comparison
            performance_improvement = ((orig_recon_loss - filt_recon_loss) / orig_recon_loss * 100).item()
            
            print(f"📊 Performance Comparison:")
            print(f"   Original data MSE: {orig_recon_loss.item():.6f}")
            print(f"   Filtered data MSE: {filt_recon_loss.item():.6f}")
            print(f"   Performance change: {performance_improvement:+.2f}%")
            
            if performance_improvement > 0:
                print("✅ LLM filtering improved model performance!")
            elif performance_improvement > -5:
                print("➡️ LLM filtering had minimal impact")
            else:
                print("⚠️ LLM filtering may need tuning")
        
        else:
            print("⚠️ No filtered data available for comparison")
    
    training_results = {
        'original_train_size': len(train_loader.dataset),
        'filtered_train_size': len(filtered_train_loader.dataset) if filtered_train_loader else 0,
        'original_test_size': len(test_loader.dataset),
        'filtered_test_size': len(filtered_test_loader.dataset) if filtered_test_loader else 0,
        'filter_config': filter_config,
        'performance_improvement': performance_improvement if 'performance_improvement' in locals() else None
    }
    
    return training_results, filtered_train_loader, filtered_test_loader

# Advanced LLM filtering strategies
class AdvancedLLMFilter:
    """
    Advanced LLM filtering with multiple strategies and adaptive thresholds.
    """
    
    def __init__(self, strategies=['statistical', 'semantic', 'confidence'], adaptive=True):
        self.strategies = strategies
        self.adaptive = adaptive
        self.performance_history = []
    
    def multi_strategy_filter(self, images, labels, model=None):
        """
        Apply multiple filtering strategies and combine results.
        """
        filter_results = {}
        
        # Statistical filtering
        if 'statistical' in self.strategies:
            stat_filter = LLMOutlierFilter(
                model_type='local',
                outlier_threshold=0.2,
                confidence_threshold=0.5
            )
            _, _, stat_results = stat_filter.filter_batch(images, labels)
            filter_results['statistical'] = stat_results
        
        # Semantic filtering (stricter)
        if 'semantic' in self.strategies:
            sem_filter = LLMOutlierFilter(
                model_type='local',
                outlier_threshold=0.1,
                confidence_threshold=0.7
            )
            _, _, sem_results = sem_filter.filter_batch(images, labels)
            filter_results['semantic'] = sem_results
        
        # Confidence-based filtering
        if 'confidence' in self.strategies:
            conf_filter = LLMOutlierFilter(
                model_type='local',
                outlier_threshold=0.3,
                confidence_threshold=0.8
            )
            _, _, conf_results = conf_filter.filter_batch(images, labels)
            filter_results['confidence'] = conf_results
        
        # Combine strategies (intersection of keep_indices)
        if filter_results:
            all_keep_indices = [set(results['keep_indices']) for results in filter_results.values()]
            combined_keep_indices = list(all_keep_indices[0].intersection(*all_keep_indices[1:]))
            
            filtered_images = images[combined_keep_indices] if combined_keep_indices else images[:0]
            filtered_labels = labels[combined_keep_indices] if combined_keep_indices else labels[:0]
            
            print(f"🔄 Multi-strategy filtering: {len(images)} → {len(filtered_images)} samples")
            
            return filtered_images, filtered_labels, {
                'combined_keep_indices': combined_keep_indices,
                'strategy_results': filter_results,
                'strategies_used': self.strategies
            }
        
        return images, labels, {'status': 'no_filtering_applied'}
    
    def adaptive_threshold_tuning(self, images, labels, model, target_retention=0.85):
        """
        Automatically tune filtering thresholds based on model performance.
        """
        print(f"🎛️ Auto-tuning thresholds for {target_retention*100:.0f}% data retention...")
        
        best_threshold = 0.15
        best_performance = float('-inf')
        
        # Test different threshold values
        for threshold in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]:
            filter_system = LLMOutlierFilter(
                model_type='local',
                outlier_threshold=threshold,
                confidence_threshold=0.6
            )
            
            filt_imgs, filt_lbls, results = filter_system.filter_batch(images, labels)
            retention_rate = len(filt_imgs) / len(images)
            
            # Check if retention rate is close to target
            if abs(retention_rate - target_retention) < 0.1 and len(filt_imgs) > 0:
                # Evaluate performance on this filtered subset
                with torch.no_grad():
                    filt_imgs_gpu = filt_imgs.to(device)
                    outputs = model(filt_imgs_gpu)
                    performance = -F.mse_loss(filt_imgs_gpu, outputs[7]).item()  # Negative MSE (higher is better)
                
                if performance > best_performance:
                    best_performance = performance
                    best_threshold = threshold
        
        print(f"✅ Optimal threshold found: {best_threshold} (performance: {-best_performance:.6f})")
        
        return best_threshold

print("🚀 Advanced LLM filtering integration ready!")
print("📋 Available functions:")
print("   - create_filtered_dataloader(): Create DataLoader with filtered data")
print("   - train_vae_with_llm_filtering(): Train VAE with filtering comparison")
print("   - AdvancedLLMFilter: Multi-strategy filtering with adaptive thresholds")
print("\n💡 Ready to enhance VAE training with intelligent outlier filtering!")

In [ ]:
# 🎯 COMPLETE PIPELINE DEMONSTRATION: VAE TRAINING WITH LLM FILTERING

print("🚀 Starting complete LLM-filtered VAE training demonstration...")
print("=" * 60)

# Load model and data
from affine_autoencoder_shared import load_autoencoder_from_file

# Use the trained model
model = load_autoencoder_from_file(
    '/Users/pmaksym/Library/CloudStorage/Box-Box/Code/adversAE/adversae/vae_training/affine/autoencoder_model_20250721_014134.pth',
    '/Users/pmaksym/Library/CloudStorage/Box-Box/Code/adversAE/adversae/vae_training/affine/autoencoder_metadata_20250721_014134.json'
)

if model:
    print("✅ Model loaded successfully!")
    
    # Apply comprehensive LLM filtering to the training pipeline
    try:
        training_results, filtered_train_loader, filtered_test_loader = train_vae_with_llm_filtering(
            model, train_loader, test_loader, {}
        )
        
        print("\n🎉 COMPLETE PIPELINE RESULTS:")
        print("=" * 40)
        print(f"📊 Dataset sizes:")
        print(f"   Original training: {training_results['original_train_size']:,} samples")
        print(f"   Filtered training: {training_results['filtered_train_size']:,} samples")
        print(f"   Original test: {training_results['original_test_size']:,} samples")
        print(f"   Filtered test: {training_results['filtered_test_size']:,} samples")
        
        if training_results['performance_improvement'] is not None:
            print(f"\n🎯 Performance Impact:")
            print(f"   Model improvement: {training_results['performance_improvement']:+.2f}%")
            
            if training_results['performance_improvement'] > 5:
                print("   🌟 Significant improvement detected!")
            elif training_results['performance_improvement'] > 0:
                print("   ✅ Positive improvement")
            else:
                print("   ⚠️ Consider threshold tuning")
        
        print(f"\n⚙️ Filter configuration used:")
        for key, value in training_results['filter_config'].items():
            print(f"   {key}: {value}")
        
        # Demonstrate advanced filtering strategies
        print("\n🧠 Testing advanced multi-strategy filtering...")
        
        advanced_filter = AdvancedLLMFilter(
            strategies=['statistical', 'semantic', 'confidence'],
            adaptive=True
        )
        
        # Get a sample batch
        sample_batch = next(iter(train_loader))
        sample_images, sample_labels = sample_batch[0][:32], sample_batch[1][:32]
        
        # Apply multi-strategy filtering
        multi_filtered_imgs, multi_filtered_lbls, multi_results = advanced_filter.multi_strategy_filter(
            sample_images, sample_labels, model
        )
        
        print(f"   Multi-strategy result: {len(sample_images)} → {len(multi_filtered_imgs)} samples")
        
        # Test adaptive threshold tuning
        if len(sample_images) > 0:
            optimal_threshold = advanced_filter.adaptive_threshold_tuning(
                sample_images, sample_labels, model, target_retention=0.80
            )
            print(f"   Optimal threshold for 80% retention: {optimal_threshold}")
        
        print("\n🎊 PIPELINE DEMONSTRATION COMPLETE!")
        print("✨ Your VAE training pipeline now includes intelligent LLM-based outlier filtering!")
        print("🔧 Customize filter parameters based on your specific data requirements.")
        
    except Exception as e:
        print(f"❌ Pipeline demonstration error: {str(e)}")
        print("💡 This may be expected if training with limited data samples.")
        print("🔧 Adjust sample_limit parameters in create_filtered_dataloader() for your dataset size.")

else:
    print("⚠️ Model not loaded. Please ensure the model files exist.")
    print("💡 You can still test the LLM filtering components independently.")